In [1]:
from sqlalchemy import create_engine, inspect, text
import sqlglot

In [2]:
engine = create_engine("sqlite:///../data/db/construction.db")

In [3]:
inspector = inspect(engine)

# Получить все таблицы
tables = inspector.get_table_names()
tables

['contractors', 'objects', 'progress', 'works']

In [121]:
# Для каждой таблицы вывести колонки
for table_name in tables:
    print(f"\n▶️ Таблица: {table_name}")
    columns = inspector.get_columns(table_name)
    for col in columns:
        print(f"  • {col['name']} | {col['type']} | nullable={col['nullable']}")

    # Внешние ключи
    fks = inspector.get_foreign_keys(table_name)
    for fk in fks:
        print(f"  ↳ FK: {fk['constrained_columns']} → {fk['referred_table']}")

    # Индексы
    indexes = inspector.get_indexes(table_name)
    for idx in indexes:
        print(f"  🔑 INDEX: {idx['name']} ({', '.join(idx['column_names'])})")


▶️ Таблица: contractors
  • id | INTEGER | nullable=True
  • name | TEXT | nullable=False
  • work_id | INTEGER | nullable=False
  ↳ FK: ['work_id'] → works

▶️ Таблица: objects
  • id | INTEGER | nullable=True
  • name | TEXT | nullable=False
  • city | TEXT | nullable=False
  • budget | REAL | nullable=False

▶️ Таблица: progress
  • id | INTEGER | nullable=True
  • work_id | INTEGER | nullable=False
  • plan_vol | REAL | nullable=False
  • fact_vol | REAL | nullable=False
  • date | TEXT | nullable=False
  ↳ FK: ['work_id'] → works

▶️ Таблица: works
  • id | INTEGER | nullable=True
  • object_id | INTEGER | nullable=False
  • work_type | TEXT | nullable=False
  • unit | TEXT | nullable=False
  ↳ FK: ['object_id'] → objects


In [105]:
with engine.connect() as conn:
    query = """
    SELECT * 
    FROM works
    JOIN contractors ON works.id = contractors.work_id
    JOIN objects ON works.object_id = objects.id
    JOIN progress ON works.id = progress.work_id
    WHERE 
        objects.name = 'ЖК Панорама 23' AND 
        objects.city = 'Санкт-Петербург' AND 
        progress.plan_vol > progress.fact_vol AND 
        contractors.name = 'ООО Новый Век'
    """

    result = conn.execute(text(query))

print(result.keys())
result.fetchall()

RMKeyView(['id', 'object_id', 'work_type', 'unit', 'id', 'name', 'work_id', 'id', 'name', 'city', 'budget', 'id', 'work_id', 'plan_vol', 'fact_vol', 'date'])


[(63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 277, 63, 271.79, 124.19, '2024-01-30'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 405, 63, 281.24, 185.88, '2024-10-16'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 368, 63, 412.6, 161.62, '2024-07-08'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 204, 63, 837.28, 793.2, '2024-11-21'),
 (63, 23, 'Окраска', 'м³', 123, 'ООО Новый Век', 63, 23, 'ЖК Панорама 23', 'Санкт-Петербург', 15220227.870224476, 181, 63, 903.98, 18.56, '2024-04-16')]

In [110]:
def get_schema_from_db(inspector) -> str:
    schema_parts = {}

    for table_name in inspector.get_table_names():
        columns = [c["name"] for c in inspector.get_columns(table_name)]
        schema_parts[table_name] = ', '.join(columns)

    return schema_parts


db_schemas = get_schema_from_db(inspector)
db_schemas

{'contractors': 'id, name, work_id',
 'objects': 'id, name, city, budget',
 'progress': 'id, work_id, plan_vol, fact_vol, date',
 'works': 'id, object_id, work_type, unit'}

In [111]:
def build_system_prompt(engine) -> str:
    """Построи prompt с полным списком реальных значений из БД"""

    with engine.connect() as conn:
        # Все подрядчики
        result = conn.execute(text("SELECT DISTINCT name FROM contractors"))
        contractors = [row[0] for row in result.fetchall()]

        # Все типы работ и единицы измерения
        result = conn.execute(text("SELECT DISTINCT work_type, unit FROM works"))
        work_types_unit = [
            f"{row[0]} - {row[1]}\n"
            for row in sorted(result.fetchall(), key=lambda x: x[0])
        ]

        # Все города
        result = conn.execute(text("SELECT DISTINCT city FROM objects"))
        cities = [row[0] for row in result.fetchall()]

    contractors_str = ", ".join([f"{c}" for c in contractors])
    work_types_str = "".join([f"{w}" for w in work_types_unit])
    cities_str = ", ".join([f"{c}" for c in cities])

    return contractors_str, work_types_str, cities_str


contractors_str, work_types_str, cities_str = build_system_prompt(engine)
contractors_str, work_types_str, cities_str

('ПАО МегаСтрой, ЗАО Качественно, ЗАО Электро-строй, ООО Надежный Строитель, ООО РазноРабота, АО Строймонтаж, АО Фундамент, ООО БазовыеРаботы, ООО ТехСтрой, ООО СтройМастер, ООО Новый Век, ООО Быстро-строй, ПАО Конструкция, ЗАО Строящий Лучше, АО Профессионал',
 'Вентиляция - пог.м\nВентиляция - шт\nВентиляция - м²\nВентиляция - м³\nВентиляция - комплект\nВентиляция - км\nВентиляция - кв.м\nВентиляция - тонн\nВнешняя отделка - комплект\nВнешняя отделка - м³\nВнешняя отделка - кв.м\nВнешняя отделка - м²\nВнешняя отделка - км\nВнешняя отделка - шт\nВнутренняя отделка - м²\nВнутренняя отделка - шт\nВнутренняя отделка - км\nВнутренняя отделка - комплект\nВнутренняя отделка - тонн\nВнутренняя отделка - м³\nВнутренняя отделка - кв.м\nВодоснабжение - км\nВодоснабжение - м³\nВодоснабжение - комплект\nВодоснабжение - тонн\nВодоснабжение - м²\nВозведение стен - шт\nВозведение стен - комплект\nВозведение стен - км\nВозведение стен - тонн\nВозведение стен - м³\nВозведение стен - м²\nЗемляные работ

LLM

In [112]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openrouter_api_key = os.getenv("OPEN_ROUTE_API_KEY")
if not openrouter_api_key:
    raise ValueError("OPEN_ROUTE_API_KEY environment variable not set.")

In [122]:
SYSTEM_PROMPT = f"""
🔧 ЗАДАЧА: Преобразуй естественный запрос в SQL. Следуй всем правилам ровно.

📊 СТРУКТУРА БД (таблицы и колонки):
{db_schemas}

📝 ДОСТУПНЫЕ ЗНАЧЕНИЯ:
- Подрядчики: {contractors_str}
- Типы работ (с единицами): {work_types_str}
- Города: {cities_str}

⚠️ КРИТИЧЕСКИ ВАЖНЫЕ ПРАВИЛА:

1. Используй ТОЛЬКО точные значения из списка выше!
   ✗ НЕПРАВИЛЬНО: WHERE city LIKE '%Петербург%'
   ✓ ПРАВИЛЬНО: WHERE city = 'Санкт-Петербург'

2. Для связей между таблицами используй INNER JOIN по FK:
   ✓ ПРАВИЛЬНО: FROM works JOIN contractors ON works.id = contractors.work_id
   ✗ НЕПРАВИЛЬНО: FROM works, contractors WHERE works.id = contractors.work_id

3. АГРЕГАЦИЯ: Если есть SUM/COUNT - ОБЯЗАТЕЛЬНО GROUP BY unit и другие неаггрегированные поля!
   ✗ НЕПРАВИЛЬНО: SELECT contractor, SUM(plan_vol) FROM works GROUP BY contractor
   ✓ ПРАВИЛЬНО: SELECT contractor, unit, SUM(plan_vol) FROM works GROUP BY contractor, unit

4. Всегда выбирай только нужные колонки, не SELECT *

ВЫХОД: ТОЛЬКО SQL без markdown, без объяснений!
"""

In [123]:
def query_llm(query: str, system_prompt: str) -> str:
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=openrouter_api_key,
    )

    completion = client.chat.completions.create(
        model = "meta-llama/llama-3.3-70b-instruct",
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query},
        ],
    )
    return completion.choices[0].message.content

In [135]:
# Запрос 1: Фильтрация по нескольким параметрам (как исходный пример)
query_1 = """
Покажи все объекты в Петербурге, для подрядчика - Новый Век, по работам связанным с покраской. 
Результат должен включать город, название объекта, имя подрядчика, название работы, ед.изм, 
плановый и фактический объем работ.
"""

# Запрос 2: Совпадение план vs факт (условие сравнения)
query_2 = """
Найди все работы, где фактический объем меньше планового объема.
Результат должен включать название объекта, тип работы, единицу измерения, плановый и фактический объемы.
"""

# Запрос 3: Агрегация и группировка
query_3 = """
Каков общий плановый объем работ по каждому подрядчику?
Результат должен содержать имя подрядчика и сумму всех плановых объемов его работ.
"""

# Запрос 4: Одна конкретная сущность (объект)
query_4 = """
Покажи все работы для объекта ЖК Панорама 23.
Результат включает название работы, тип работы, единицу измерения, подрядчика и текущий статус прогресса.
"""

# Запрос 5: Поиск по типу работ без привязки к конкретному подрядчику
query_5 = """
Покажи все работы по типу "Отделка" по всем объектам в Москве.
Результат должен содержать название объекта, подрядчика, единицу измерения и объемы работ.
"""

Формирования, с помощью LLM, SQL запроса из текстового описания задачи

In [132]:
sql_query_llm = query_llm(query=query_2, system_prompt=SYSTEM_PROMPT)
sql_query_llm

'SELECT o.name, w.work_type, w.unit, p.plan_vol, p.fact_vol \nFROM progress p \nJOIN works w ON p.work_id = w.id \nJOIN objects o ON w.object_id = o.id \nWHERE p.fact_vol < p.plan_vol \nGROUP BY o.name, w.work_type, w.unit, p.plan_vol, p.fact_vol'

In [133]:
def execute_sql_query(engine, query: str):
    """Выполняет SQL запрос и возвращает результат"""
    try:
        with engine.connect() as conn:
            sql_query_llm = query_llm(query=query, system_prompt=SYSTEM_PROMPT)
            check_sql_query = sqlglot.transpile(sql_query_llm)
            query = check_sql_query[0]
            result = conn.execute(text(query))

        return result.fetchall()
    except Exception as e:
        print(f"Ошибка при выполнении SQL запроса: {e}")
        return f"Ошибка при выполнении SQL запроса: {e}"

In [136]:
execute_sql_query(engine, query_3)

[('АО Профессионал', 32680.76),
 ('АО Строймонтаж', 29360.29),
 ('АО Фундамент', 35843.7),
 ('ЗАО Качественно', 32052.14),
 ('ЗАО Строящий Лучше', 31631.93),
 ('ЗАО Электро-строй', 32658.9),
 ('ООО БазовыеРаботы', 37424.67),
 ('ООО Быстро-строй', 28394.69),
 ('ООО Надежный Строитель', 36610.72),
 ('ООО Новый Век', 25154.79),
 ('ООО РазноРабота', 43132.05),
 ('ООО СтройМастер', 30503.39),
 ('ООО ТехСтрой', 46253.41),
 ('ПАО Конструкция', 31130.35),
 ('ПАО МегаСтрой', 23298.45)]